# LeetCode 2706 — Buy Two Chocolates bằng Python

Link đề: https://leetcode.com/problems/buy-two-chocolates/ · Độ khó: **Easy** · Chủ đề: Array, Greedy, Sorting

Skeleton LeetCode:

```python
class Solution:
    def buyChoco(self, prices: List[int], money: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: chọn **đúng hai thanh sô-cô-la khác nhau** sao cho **còn dư nhiều tiền nhất** mà **không bị âm**; nếu không thể thì trả về đúng số tiền ban đầu.
2. Nhận ra bản chất **tham lam (greedy)**: muốn dư nhiều nhất thì phải trả **ít nhất** ⇒ chọn **hai thanh rẻ nhất**.
3. Dùng `sorted`, `min`, `float("inf")`, `itertools.combinations`, `heapq.nsmallest`.
4. Viết 4 cách: thử mọi cặp (brute force), sắp xếp rồi lấy hai số nhỏ nhất, một lượt duyệt giữ hai giá trị nhỏ nhất, và `heapq.nsmallest`.
5. Chứng minh vì sao hai số nhỏ nhất là tối ưu, và đối chiếu các cách bằng test ngẫu nhiên có seed.
6. Tránh các lỗi quen thuộc: không kiểm tra tiền âm, dùng cùng một thanh hai lần, cập nhật sai hai biến nhỏ nhất.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `prices` — giá của các thanh sô-cô-la trong cửa hàng — và một số nguyên `money` — số tiền bạn đang có.

Bạn phải mua **đúng hai thanh** sô-cô-la sao cho **vẫn còn dư tiền không âm**. Bạn muốn **tối thiểu hoá tổng giá** hai thanh đã mua (tức là tối đa hoá tiền thừa).

Trả về **số tiền còn lại** sau khi mua. Nếu **không thể** mua hai thanh mà không bị nợ (tổng giá hai thanh rẻ nhất vẫn lớn hơn `money`), trả về **`money`** (không mua gì cả).

| Ví dụ | prices | money | Output | Giải thích |
|---|---|---|---|---|
| 1 | `[1,2,2]` | `3` | `0` | mua thanh giá `1` và `2`: tổng `3`, còn `3 - 3 = 0` |
| 2 | `[3,2,3]` | `3` | `3` | hai thanh rẻ nhất `2 + 3 = 5 > 3` ⇒ không mua được, trả `money = 3` |

Constraints: `2 <= prices.length <= 50`; `1 <= prices[i] <= 100`; `1 <= money <= 100`.

In [1]:
prices, money = [1, 2, 2], 3
print("ví dụ 1: 1 + 2 =", 1 + 2, "-> còn", money - (1 + 2))
prices, money = [3, 2, 3], 3
print("ví dụ 2: 2 + 3 =", 2 + 3, "> money =", money, "-> không mua được, trả", money)

ví dụ 1: 1 + 2 = 3 -> còn 0
ví dụ 2: 2 + 3 = 5 > money = 3 -> không mua được, trả 3


## 3. Kiến thức nền

### 3.1. `sorted` và lát cắt `[:2]`

`sorted(prices)` trả về **list mới** đã sắp tăng dần (không sửa `prices`). `a.sort()` thì sửa tại chỗ. Với list đã sắp, `a[0]` là nhỏ nhất, `a[1]` nhỏ thứ hai, `a[:2]` lấy hai phần tử đầu.

In [2]:
prices = [3, 2, 3, 1]
a = sorted(prices)
print(a, "| gốc không đổi:", prices)     # [1, 2, 3, 3] | [3, 2, 3, 1]
print(a[0], a[1], a[:2], sum(a[:2]))      # 1 2 [1, 2] 3

[1, 2, 3, 3] | gốc không đổi: [3, 2, 3, 1]
1 2 [1, 2] 3


### 3.2. Duyệt mọi cặp: hai vòng lặp và `itertools.combinations`

Cặp hai phần tử khác **chỉ số** `(i, j)` với `i < j`: dùng hai vòng `for`, hoặc `combinations(prices, 2)` (sinh mọi cặp mà không lặp lại). Chú ý: hai thanh khác chỉ số có thể **cùng giá** (ví dụ `[2, 2]`), đó vẫn là hai thanh khác nhau.

In [3]:
from itertools import combinations

prices = [1, 2, 2]
for i in range(len(prices)):
    for j in range(i + 1, len(prices)):
        print(f"i={i}, j={j}: {prices[i]} + {prices[j]} = {prices[i] + prices[j]}")

print(list(combinations(prices, 2)))      # [(1, 2), (1, 2), (2, 2)]  -> 3 cặp

i=0, j=1: 1 + 2 = 3
i=0, j=2: 1 + 2 = 3
i=1, j=2: 2 + 2 = 4
[(1, 2), (1, 2), (2, 2)]


### 3.3. `float("inf")`: "lớn vô cực" làm giá trị khởi đầu

Khi tìm **nhỏ nhất**, ta khởi tạo biến bằng một số **lớn hơn mọi dữ liệu** để lần so sánh đầu tiên luôn thắng. `float("inf")` là vô cực của Python; mọi số hữu hạn đều nhỏ hơn nó.

In [4]:
INF = float("inf")
print(5 < INF, 10**9 < INF, INF > -INF)    # True True True
smallest = INF
for x in [4, 7, 2]:
    if x < smallest:
        smallest = x
print(smallest)                             # 2

True True True
2


### 3.4. `heapq.nsmallest(k, iterable)` và biểu thức điều kiện

- `heapq.nsmallest(2, a)` trả list gồm **hai phần tử nhỏ nhất** (đã sắp tăng dần), không cần sắp cả mảng.
- Biểu thức điều kiện: `x if điều_kiện else y` — gọn hơn viết `if/else` nhiều dòng.

In [5]:
import heapq

a = [5, 2, 4, 1, 3]
print(heapq.nsmallest(2, a))               # [1, 2]
left, money = 7, 10
print(left if left >= 0 else money)         # 7
print(-1 if -1 >= 0 else money)             # 10

[1, 2]
7
10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Mục tiêu "dư nhiều tiền nhất" = "**trả ít nhất**". Mua hai thanh thì ta phải trả **giá thanh 1 + giá thanh 2**. Vậy ta muốn hai thanh có **tổng nhỏ nhất**. Rất tự nhiên: **chọn hai thanh rẻ nhất**!

Sau đó chỉ có hai khả năng:

- Tổng hai thanh rẻ nhất `≤ money` ⇒ mua được, **tiền còn lại = `money − tổng`**.
- Tổng `> money` ⇒ **mọi** cặp thanh khác đều đắt hơn (vì đây đã là cặp rẻ nhất) ⇒ không mua được ⇒ trả về `money`.

Tính tay `prices = [3,2,3], money = 3`: hai thanh rẻ nhất `2` và `3`, tổng `5 > 3` ⇒ trả `3`.
Tính tay `prices = [1,2,2], money = 3`: rẻ nhất `1` và `2`, tổng `3 ≤ 3` ⇒ còn `0`.

In [6]:
def tinh_tay(prices, money):
    a = sorted(prices)
    cost = a[0] + a[1]
    return money - cost if cost <= money else money

print(tinh_tay([1, 2, 2], 3), tinh_tay([3, 2, 3], 3))    # 0 3

0 3


## 5. Cách 1 — Brute Force / baseline: thử mọi cặp

**Ý tưởng:** đề nói "mua hai thanh sao cho còn dư không âm và dư **nhiều nhất**". Ta thử **mọi cặp** `(i, j)`, `i < j`: nếu `prices[i] + prices[j] <= money` thì tiền dư là `money - (prices[i] + prices[j])`; giữ lại **dư lớn nhất**. Nếu không cặp nào mua được thì trả `money`.

Cách này gần như dịch nguyên văn đề, nên là một **oracle** tốt.

In [7]:
from typing import List

def buy_choco_pairs(prices: List[int], money: int) -> int:
    best = money                          # không mua được thì giữ nguyên money
    found = False
    n = len(prices)
    for i in range(n):
        for j in range(i + 1, n):
            cost = prices[i] + prices[j]
            if cost <= money:
                left = money - cost
                if not found or left > best:
                    best = left
                    found = True
    return best

print(buy_choco_pairs([1, 2, 2], 3))    # 0
print(buy_choco_pairs([3, 2, 3], 3))    # 3

0
3


**Độ phức tạp:** `n(n-1)/2` cặp ⇒ Time `O(n²)` (tối đa ~1225 cặp với `n = 50`), Space `O(1)`.

**Nút thắt:** ta xét cả những cặp rõ ràng không tối ưu (có thanh đắt). Chỉ cần **hai thanh rẻ nhất** là đủ ⇒ không cần so sánh mọi cặp.

## 6. Ý tưởng thuật toán / bước đột phá

Tổng `prices[i] + prices[j]` nhỏ nhất khi `prices[i]` và `prices[j]` là **hai phần tử nhỏ nhất** của mảng. Có thể thấy bằng **lập luận hoán đổi** (exchange argument): nếu một cặp tối ưu chứa phần tử không thuộc hai nhỏ nhất, đổi nó với phần tử nhỏ nhất còn thiếu chỉ làm tổng giảm hoặc không đổi.

Vậy bài toán rút gọn thành: **tìm hai số nhỏ nhất** của mảng. Có nhiều cách:

1. Sắp xếp rồi lấy `a[0] + a[1]` (`O(n log n)`).
2. Duyệt một lượt, giữ hai giá trị nhỏ nhất `min1 ≤ min2` (`O(n)`).
3. `heapq.nsmallest(2, prices)`.

In [8]:
prices = [5, 2, 4, 1, 3]
min1 = min2 = float("inf")
for x in prices:
    if x < min1:
        min1, min2 = x, min1     # x thành nhỏ nhất, nhỏ nhất cũ tụt xuống nhỏ nhì
    elif x < min2:
        min2 = x                 # x chen vào làm nhỏ nhì
    print(f"x={x}: min1={min1}, min2={min2}")

x=5: min1=5, min2=inf
x=2: min1=2, min2=5
x=4: min1=2, min2=4
x=1: min1=1, min2=2
x=3: min1=1, min2=2


## 7. Các cách tiếp theo

Ngoài Cách 1 (mọi cặp), ta xem 3 cách: (2) sắp xếp, (3) một lượt giữ hai số nhỏ nhất, (4) `heapq.nsmallest`.

### Cách 2 — Sắp xếp rồi lấy hai số đầu

Giải thích từng đoạn:

- `a = sorted(prices)`: bản sao đã sắp tăng dần (không đổi `prices` gốc).
- `cost = a[0] + a[1]`: tổng hai thanh rẻ nhất (mảng có ít nhất 2 phần tử nên luôn hợp lệ).
- `return money - cost if cost <= money else money`: nếu mua được thì trả tiền dư, ngược lại trả `money`.

In [9]:
def buy_choco_sort(prices: List[int], money: int) -> int:
    a = sorted(prices)
    cost = a[0] + a[1]
    return money - cost if cost <= money else money

print(buy_choco_sort([1, 2, 2], 3), buy_choco_sort([3, 2, 3], 3))

0 3


In [10]:
from typing import List

class SolutionSort:
    def buyChoco(self, prices: List[int], money: int) -> int:
        a = sorted(prices)
        cost = a[0] + a[1]
        return money - cost if cost <= money else money

print(SolutionSort().buyChoco([1, 2, 2], 3))

0


**Độ phức tạp:** Time `O(n log n)` do `sorted`; Space `O(n)` (bản sao). Nếu dùng `prices.sort()` tại chỗ thì Space phụ `O(1)` nhưng làm thay đổi mảng đầu vào.

### Cách 3 — Một lượt duyệt, giữ hai số nhỏ nhất `min1`, `min2`

Giải thích từng đoạn:

- `min1 = min2 = inf`: chưa có số nào.
- Với mỗi `x`: nếu `x < min1` thì `x` là nhỏ nhất mới và nhỏ nhất cũ trở thành nhỏ nhì: `min1, min2 = x, min1`. Ngược lại nếu `x < min2` thì `x` chỉ chen vào làm nhỏ nhì.
- Dùng `elif` (không phải hai `if` độc lập) để không cập nhật hai lần với cùng một `x`.
- Cuối cùng `cost = min1 + min2`.

In [11]:
def buy_choco_one_pass(prices: List[int], money: int) -> int:
    inf = float("inf")
    min1 = min2 = inf
    for x in prices:
        if x < min1:
            min1, min2 = x, min1
        elif x < min2:
            min2 = x
    cost = min1 + min2
    return money - cost if cost <= money else money

print(buy_choco_one_pass([1, 2, 2], 3), buy_choco_one_pass([3, 2, 3], 3))

0 3


In [12]:
from typing import List

class SolutionOnePass:
    def buyChoco(self, prices: List[int], money: int) -> int:
        min1 = min2 = float("inf")
        for x in prices:
            if x < min1:
                min1, min2 = x, min1
            elif x < min2:
                min2 = x
        cost = min1 + min2
        return money - cost if cost <= money else money

print(SolutionOnePass().buyChoco([3, 2, 3], 3))

3


**Độ phức tạp:** một lượt ⇒ Time `O(n)`, Space `O(1)`. Đây là cách tối ưu về mặt thuật toán.

Lưu ý: dùng `elif x < min2` (nghiêm ngặt) vẫn đúng khi có **giá trùng nhau** như `[2, 2]`: lần đầu `min1 = 2, min2 = inf`; `x = 2` không `< min1`, nhưng `2 < inf` ⇒ `min2 = 2`.

### Cách 4 — `heapq.nsmallest(2, prices)`

Thư viện chuẩn có sẵn hàm lấy `k` phần tử nhỏ nhất. Với `k = 2`: `a, b = heapq.nsmallest(2, prices)`; `cost = a + b`. Về bản chất tương đương cách 3, nhưng ngắn.

In [13]:
import heapq

def buy_choco_heap(prices: List[int], money: int) -> int:
    a, b = heapq.nsmallest(2, prices)
    cost = a + b
    return money - cost if cost <= money else money

print(buy_choco_heap([1, 2, 2], 3), buy_choco_heap([3, 2, 3], 3))

0 3


In [14]:
import heapq
from typing import List

class SolutionHeap:
    def buyChoco(self, prices: List[int], money: int) -> int:
        cost = sum(heapq.nsmallest(2, prices))
        return money - cost if cost <= money else money

print(SolutionHeap().buyChoco([1, 2, 2], 3))

0


**Độ phức tạp:** `nsmallest(2, n phần tử)` chạy `O(n log 2) = O(n)`; Space `O(1)` phụ (heap kích thước 2). Có thể coi như `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 với `prices = [5, 2, 4, 1, 3]`, `money = 10`:

| `x` | `x < min1`? | `x < min2`? | `min1` sau | `min2` sau |
|---|---|---|---|---|
| (đầu) | | | `inf` | `inf` |
| 5 | có | | 5 | `inf` |
| 2 | có | | 2 | 5 |
| 4 | không | có (`4 < 5`) | 2 | 4 |
| 1 | có | | 1 | 2 |
| 3 | không | không (`3 < 2` sai) | 1 | 2 |

`cost = 1 + 2 = 3 ≤ 10` ⇒ trả `10 - 3 = 7`.

In [15]:
def trace_one_pass(prices, money):
    inf = float("inf")
    min1 = min2 = inf
    print(f"đầu: min1={min1}, min2={min2}")
    for x in prices:
        if x < min1:
            action = f"{x} < min1 -> min1={x}, min2={min1}"
            min1, min2 = x, min1
        elif x < min2:
            action = f"{x} < min2 -> min2={x}"
            min2 = x
        else:
            action = "bỏ qua"
        print(f"x={x}: {action:<28} => min1={min1}, min2={min2}")
    cost = min1 + min2
    ans = money - cost if cost <= money else money
    print(f"cost={cost}, money={money} -> đáp án {ans}")
    return ans

trace_one_pass([5, 2, 4, 1, 3], 10)

đầu: min1=inf, min2=inf
x=5: 5 < min1 -> min1=5, min2=inf => min1=5, min2=inf
x=2: 2 < min1 -> min1=2, min2=5   => min1=2, min2=5
x=4: 4 < min2 -> min2=4           => min1=2, min2=4
x=1: 1 < min1 -> min1=1, min2=2   => min1=1, min2=2
x=3: bỏ qua                       => min1=1, min2=2
cost=3, money=10 -> đáp án 7


7

In [16]:
# Trace với giá trùng nhau và trường hợp không mua được
trace_one_pass([2, 2, 2], 4)
print()
trace_one_pass([3, 2, 3], 3)

đầu: min1=inf, min2=inf
x=2: 2 < min1 -> min1=2, min2=inf => min1=2, min2=inf
x=2: 2 < min2 -> min2=2           => min1=2, min2=2
x=2: bỏ qua                       => min1=2, min2=2
cost=4, money=4 -> đáp án 0

đầu: min1=inf, min2=inf
x=3: 3 < min1 -> min1=3, min2=inf => min1=3, min2=inf
x=2: 2 < min1 -> min1=2, min2=3   => min1=2, min2=3
x=3: bỏ qua                       => min1=2, min2=3
cost=5, money=3 -> đáp án 3


3

## 9. Code nộp LeetCode

Dùng Cách 3 (một lượt), đúng class `Solution` và method `buyChoco`.

In [17]:
from typing import List

class Solution:
    def buyChoco(self, prices: List[int], money: int) -> int:
        min1 = min2 = float("inf")
        for x in prices:
            if x < min1:
                min1, min2 = x, min1
            elif x < min2:
                min2 = x
        cost = min1 + min2
        return money - cost if cost <= money else money

In [18]:
print(Solution().buyChoco([1, 2, 2], 3))    # 0
print(Solution().buyChoco([3, 2, 3], 3))    # 3

0
3


## 10. Vì sao thuật toán đúng

**Ý 1 — Hai số nhỏ nhất cho tổng nhỏ nhất.** Sắp mảng thành `a_0 ≤ a_1 ≤ ... ≤ a_{n-1}`. Một cặp hai thanh khác nhau tương ứng hai chỉ số `p < q` trong mảng đã sắp, có tổng `a_p + a_q`. Vì `p ≥ 0` nên `a_p ≥ a_0`; vì `q > p ≥ 0` nên `q ≥ 1` và `a_q ≥ a_1`. Cộng lại: `a_p + a_q ≥ a_0 + a_1`. Cặp `(0, 1)` đạt đúng giá trị này nên **`a_0 + a_1` là tổng nhỏ nhất** (kể cả khi có giá trùng nhau).

**Ý 2 — Tiền dư lớn nhất.** `tiền dư = money − tổng`; tổng nhỏ nhất ⇒ tiền dư lớn nhất.

**Ý 3 — Điều kiện không âm.** Nếu `a_0 + a_1 ≤ money` thì cặp này hợp lệ và tối ưu. Nếu `a_0 + a_1 > money` thì mọi cặp khác có tổng `≥ a_0 + a_1 > money` nên không cặp nào hợp lệ ⇒ trả `money` đúng như đề.

**Ý 4 — Cách 3 tìm đúng hai nhỏ nhất.** Invariant: sau khi xét một số phần tử, `min1` là nhỏ nhất, `min2` là nhỏ thứ hai (tính cả trùng). Khi `x < min1`: `x` thành nhỏ nhất, nhỏ nhất cũ thành nhỏ nhì. Khi `min1 ≤ x < min2`: `x` thành nhỏ nhì. Còn lại không đổi. Vì `n ≥ 2` nên cuối cùng cả `min1, min2` hữu hạn.

## 11. Phân tích Time / Space Complexity

Gọi `n = len(prices)` (tối đa 50).

| Cách | Time | Space |
|---|---|---|
| 1. mọi cặp | `O(n²)` | `O(1)` |
| 2. sắp xếp | `O(n log n)` | `O(n)` (bản sao) |
| 3. một lượt hai số nhỏ nhất | `O(n)` | `O(1)` |
| 4. `heapq.nsmallest(2, ...)` | `O(n)` | `O(1)` |

Cận dưới là `O(n)` vì phải đọc mọi phần tử ít nhất một lần; Cách 3 và 4 đạt cận đó. Vì `prices[i] ≤ 100`, còn có thể dùng đếm bằng mảng 101 ô (`O(n + 100)`) — xem mục 15.

## 12. Bộ test

### 12.1. Ví dụ của đề

In [19]:
sols = [SolutionSort(), SolutionOnePass(), SolutionHeap(), Solution()]
funcs = [buy_choco_pairs, buy_choco_sort, buy_choco_one_pass, buy_choco_heap] + [s.buyChoco for s in sols]

for f in funcs:
    assert f([1, 2, 2], 3) == 0
    assert f([3, 2, 3], 3) == 3
print("Ví dụ của đề: tất cả cách đều đúng")

Ví dụ của đề: tất cả cách đều đúng


### 12.2. Edge case

- Mảng đúng 2 phần tử: `[1, 1], 2` → `0`; `[5, 6], 10` → `10` (11 > 10 ⇒ trả `money`).
- Hai thanh rẻ nhất trùng giá: `[2, 2, 9], 4` → `0`; `[2, 2, 9], 5` → `1`.
- Vừa đủ tiền (dư `0`) và thiếu đúng `1` đồng: `[3, 4], 7` → `0`; `[3, 4], 6` → `6`.
- Giá lớn nhất: `[100, 100], 100` → `100`; toàn `1` với `money = 100` → `98`.
- Mọi giá bằng nhau: `[5, 5, 5, 5], 10` → `0`.
- Thanh rẻ nhất nằm ở cuối: `[9, 8, 7, 1, 2], 10` → `7`.

In [20]:
cases = [
    (([1, 1], 2), 0),
    (([5, 6], 10), 10),
    (([2, 2, 9], 4), 0),
    (([2, 2, 9], 5), 1),
    (([3, 4], 7), 0),
    (([3, 4], 6), 6),
    (([100, 100], 100), 100),
    (([1] * 50, 100), 98),
    (([5, 5, 5, 5], 10), 0),
    (([9, 8, 7, 1, 2], 10), 7),
]
for f in funcs:
    for (p, m), want in cases:
        assert f(list(p), m) == want, (f, p, m, want)
print("Edge case OK:", len(cases), "trường hợp x", len(funcs), "cách")

# kiểm tra không làm thay đổi mảng đầu vào
p = [5, 2, 4, 1, 3]
before = p[:]
for s in sols:
    s.buyChoco(p, 10)
assert p == before
print("Không sửa mảng đầu vào")

Edge case OK: 10 trường hợp x 8 cách
Không sửa mảng đầu vào


### 12.3. Phản ví dụ cho cách làm sai

Hai cách sai hay gặp: "lấy thanh rẻ nhất hai lần" và "trả `money − cost` mà không kiểm tra âm". Ví dụ nhỏ để thấy cả hai sai.

In [21]:
def sai_hai_lan(prices, money):
    cost = 2 * min(prices)                  # dùng cùng một thanh hai lần
    return money - cost if cost <= money else money

def sai_khong_kiem_tra_am(prices, money):
    a = sorted(prices)
    return money - (a[0] + a[1])

print("sai_hai_lan([1,5], 3) =", sai_hai_lan([1, 5], 3), "| đúng:", Solution().buyChoco([1, 5], 3))
print("sai_khong_kiem_tra_am([3,2,3], 3) =", sai_khong_kiem_tra_am([3, 2, 3], 3), "| đúng:", Solution().buyChoco([3, 2, 3], 3))
assert sai_hai_lan([1, 5], 3) != Solution().buyChoco([1, 5], 3)
assert sai_khong_kiem_tra_am([3, 2, 3], 3) != Solution().buyChoco([3, 2, 3], 3)

sai_hai_lan([1,5], 3) = 1 | đúng: 3
sai_khong_kiem_tra_am([3,2,3], 3) = -2 | đúng: 3


### 12.4. Đối chiếu nhiều input ngẫu nhiên (có seed)

Sinh dữ liệu hợp lệ theo constraints: `n` từ 2 đến 50, `prices[i]` từ 1 đến 100, `money` từ 1 đến 100. Để có cả trường hợp mua được và không mua được, ta trộn các chế độ giá nhỏ và giá lớn. Dùng cách 1 (mọi cặp) làm oracle.

In [22]:
import random

random.seed(2706)
cnt_buy = cnt_no = 0
for _ in range(3000):
    n = random.randint(2, 50)
    hi = random.choice([5, 20, 60, 100])            # miền giá khác nhau để có nhiều tình huống
    prices = [random.randint(1, hi) for _ in range(n)]
    money = random.randint(1, 100) if random.random() < 0.5 else random.randint(1, 12)   # nhiều ca không mua được
    want = buy_choco_pairs(prices, money)
    for f in funcs:
        assert f(prices[:], money) == want, (prices, money, f)
    if want == money:
        cnt_no += 1
    else:
        cnt_buy += 1
print("OK 3000 test; mua được:", cnt_buy, "| không mua được/dư không đổi:", cnt_no)
assert cnt_buy > 100 and cnt_no > 100          # đảm bảo cả hai nhánh đều được thử

OK 3000 test; mua được: 2229 | không mua được/dư không đổi: 771


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Quên kiểm tra tiền âm

In [23]:
from typing import List

def sai_am(prices: List[int], money: int) -> int:
    a = sorted(prices)
    return money - (a[0] + a[1])             # SAI: có thể âm

print("sai :", sai_am([3, 2, 3], 3))                 # -2
print("đúng:", Solution().buyChoco([3, 2, 3], 3))    # 3
assert sai_am([3, 2, 3], 3) < 0

sai : -2
đúng: 3


**Vì sao sai:** đề yêu cầu **không nợ**. Nếu tổng hai thanh rẻ nhất lớn hơn `money` thì phải **bỏ cuộc** và trả đúng `money` (chứ không phải trả `0`, cũng không phải số âm).

### Lỗi 2: Dùng cùng một thanh hai lần

In [24]:
def sai_dung_hai_lan(prices: List[int], money: int) -> int:
    cost = min(prices) * 2                   # SAI: mua hai thanh nhưng chọn cùng một thanh rẻ nhất
    return money - cost if cost <= money else money

print("sai :", sai_dung_hai_lan([1, 5], 3))              # 1  (nghĩ là mua 1 + 1)
print("đúng:", Solution().buyChoco([1, 5], 3))           # 3  (thật sự 1 + 5 = 6 > 3)
assert sai_dung_hai_lan([1, 5], 3) != Solution().buyChoco([1, 5], 3)

sai : 1
đúng: 3


**Vì sao sai:** hai thanh phải là **hai thanh khác nhau** (hai chỉ số khác nhau). Thanh rẻ nhất chỉ có một thanh; thanh thứ hai là thanh **rẻ nhì**. (Nếu có hai thanh cùng giá rẻ nhất thì dùng cả hai là hợp lệ vì chúng là hai thanh khác nhau.)

### Lỗi 3: Cập nhật cả `min1` và `min2` bằng hai `if` độc lập

In [25]:
def sai_hai_if(prices: List[int], money: int) -> int:
    min1 = min2 = float("inf")
    for x in prices:
        if x < min1:
            min1 = x                        # SAI: min2 không nhận min1 cũ
        if x < min2:
            min2 = x                        # SAI: min2 cũng bị gán theo x -> min1 == min2
    cost = min1 + min2
    return money - cost if cost <= money else money

print("sai :", sai_hai_if([3, 2, 3], 4))                 # min1 = min2 = 2 -> cost 4 -> 0
print("đúng:", Solution().buyChoco([3, 2, 3], 4))        # 2 + 3 = 5 > 4 -> 4
assert sai_hai_if([3, 2, 3], 4) != Solution().buyChoco([3, 2, 3], 4)

sai : 0
đúng: 4


**Vì sao sai:** với hai `if` độc lập, khi `x` nhỏ hơn cả `min1` lẫn `min2` thì cả hai biến cùng bị đặt bằng `x`, nên `min1 == min2` — như thể ta mua hai lần cùng một thanh. Dùng `min1, min2 = x, min1` rồi `elif x < min2`.

### Lỗi 4: Sửa mảng đầu vào hoặc chỉ xét `a[0]` và `a[-1]`

In [26]:
def sai_dau_cuoi(prices: List[int], money: int) -> int:
    a = sorted(prices)
    cost = a[0] + a[-1]                      # SAI: cuối mảng là thanh ĐẮT nhất
    return money - cost if cost <= money else money

print("sai :", sai_dau_cuoi([1, 2, 2], 3))              # 1 + 2 = 3 -> 0 (trùng ngẫu nhiên)
print("sai :", sai_dau_cuoi([1, 2, 3], 4))              # 1 + 3 = 4 -> 0
print("đúng:", Solution().buyChoco([1, 2, 3], 4))        # 1 + 2 = 3 -> 1
assert sai_dau_cuoi([1, 2, 3], 4) != Solution().buyChoco([1, 2, 3], 4)

sai : 0
sai : 0
đúng: 1


**Vì sao sai:** nhầm `a[1]` (thứ hai nhỏ nhất) với `a[-1]` (lớn nhất). Ví dụ đề `[1, 2, 2]` trùng đúng ngẫu nhiên nên cho kết quả đúng — **vì thế phải tự thử thêm test của riêng mình**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mua/chọn **hai** phần tử sao cho **tổng nhỏ nhất**" ⇒ lấy **hai nhỏ nhất**. Tương tự "tổng lớn nhất" ⇒ hai lớn nhất.
- Chọn `k` nhỏ nhất ⇒ `sorted(...)[:k]` hoặc `heapq.nsmallest(k, ...)`.
- Hai số nhỏ nhất trong một lượt: `min1, min2 = inf, inf` và `if x < min1: min1, min2 = x, min1 elif x < min2: min2 = x`.
- Khi đề có "nếu không thể thì trả về X" luôn tự hỏi: **X là gì chính xác?** (ở đây là `money`, không phải `0` hay `-1`).
- Giá trị nhỏ (`≤ 100`) gợi ý có thể **đếm bằng mảng tần suất** — tiện khi `n` rất lớn.
- Với ví dụ đề trùng ngẫu nhiên, hãy tự nghĩ thêm test có phần tử lớn nhất xen vào giữa để phát hiện lỗi `a[-1]`.

## 15. Biến thể của bài

1. **Mua `k` thanh** để dư nhiều nhất: cộng `k` giá nhỏ nhất, kiểm tra `≤ money`.
2. **Mua nhiều nhất có thể** với `money` đồng (LeetCode 1833 Maximum Ice Cream Bars): sắp xếp rồi mua từ rẻ đến đắt đến hết tiền.
3. **Đếm số cặp** có tổng `≤ money`: sắp xếp + hai con trỏ.
4. **Hai thanh có tổng đúng bằng `money`**: Two Sum (LeetCode 1).
5. **Đếm bằng mảng tần suất** vì giá chỉ trong `1..100`: `O(n + 100)`.
6. **Chọn hai thanh tốn tiền nhiều nhất không vượt quá `money`** (tiền dư nhỏ nhất `≥ 0`): đây không còn là tham lam hai nhỏ nhất; cần sắp xếp + hai con trỏ.

In [27]:
def buy_choco_counting(prices, money):
    cnt = [0] * 101                       # giá 1..100
    for p in prices:
        cnt[p] += 1
    picked = []
    for price in range(1, 101):
        while cnt[price] > 0 and len(picked) < 2:
            picked.append(price)
            cnt[price] -= 1
        if len(picked) == 2:
            break
    cost = sum(picked)
    return money - cost if cost <= money else money

print(buy_choco_counting([1, 2, 2], 3), buy_choco_counting([3, 2, 3], 3))     # 0 3

def buy_k(prices, money, k):
    cost = sum(sorted(prices)[:k])
    return money - cost if cost <= money else money

print(buy_k([5, 2, 4, 1, 3], 10, 3))      # 1 + 2 + 3 = 6 -> còn 4

0 3
4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. mọi cặp | thử hết `(i, j)` | `O(n²)` | `O(1)` | hiểu đề, làm oracle |
| 2. sắp xếp | lấy `a[0] + a[1]` | `O(n log n)` | `O(n)` | code ngắn, `n` nhỏ |
| 3. một lượt `min1`, `min2` | giữ hai số nhỏ nhất | `O(n)` | `O(1)` | **cách nộp bài khuyến nghị** |
| 4. `heapq.nsmallest(2, ...)` | thư viện lấy hai nhỏ nhất | `O(n)` | `O(1)` | ngắn gọn, dễ mở rộng `k` |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `sorted([4, 1, 3])[:2]`, `min(5, 2) + max(5, 2)`, `float("inf") > 10**9`, `heapq.nsmallest(2, [9, 3, 7])`.

**Bài 2 (tính tay).** `prices = [4, 7, 2, 5]`, `money = 8`: hai thanh rẻ nhất là gì, trả về bao nhiêu? Đổi `money = 5` thì sao?

**Bài 3 (điền TODO).** Hoàn thành `buy_choco_todo(prices, money)` bằng sắp xếp.

In [28]:
def buy_choco_todo(prices, money):
    # TODO 1: a = sorted(prices)
    # TODO 2: cost = tổng hai phần tử đầu
    # TODO 3: nếu cost <= money trả money - cost, ngược lại trả money
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `buy_choco_one_pass2(prices, money)` bằng một lượt với `min1`, `min2` mà không nhìn Cách 3. Kiểm tra với hai ví dụ của đề.

**Bài 5 (biến thể / mini challenge).** (a) Viết `max_bars(prices, money)` (LeetCode 1833): mua **nhiều thanh nhất** có thể với `money`, trả số thanh. (b) Viết `count_pairs_affordable(prices, money)`: đếm số cặp chỉ số `i < j` có `prices[i] + prices[j] <= money` bằng sắp xếp + hai con trỏ.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `[1, 3]`; `7`; `True`; `[3, 7]`. Bài 2: hai thanh `2` và `4`, tổng `6`.
- **Mức 2 (vừa):** Bài 3: `a[0] + a[1]`, rồi biểu thức điều kiện. Bài 4: `min1 = min2 = float("inf")`, `elif x < min2`.
- **Mức 3 (sâu):** Bài 5(a): sắp xếp, cộng dồn từ nhỏ đến lớn, dừng khi vượt `money`. Bài 5(b): sắp xếp; `l, r = 0, n - 1`; nếu `a[l] + a[r] <= money` thì mọi `r' ∈ (l, r]` đều ghép được với `l` ⇒ cộng `r - l` rồi `l += 1`; ngược lại `r -= 1`.

## 19. Đáp án tham khảo cho bài tập

In [29]:
import heapq

# Bài 1
assert sorted([4, 1, 3])[:2] == [1, 3]
assert min(5, 2) + max(5, 2) == 7
assert float("inf") > 10**9
assert heapq.nsmallest(2, [9, 3, 7]) == [3, 7]

# Bài 2: [4,7,2,5], money = 8: hai rẻ nhất 2 và 4 -> 6 <= 8 -> 2; money = 5 -> 6 > 5 -> 5
assert Solution().buyChoco([4, 7, 2, 5], 8) == 2
assert Solution().buyChoco([4, 7, 2, 5], 5) == 5

# Bài 3
def buy_choco_todo_ans(prices, money):
    a = sorted(prices)
    cost = a[0] + a[1]
    return money - cost if cost <= money else money

assert buy_choco_todo_ans([1, 2, 2], 3) == 0
assert buy_choco_todo_ans([3, 2, 3], 3) == 3

# Bài 4
def buy_choco_one_pass2(prices, money):
    min1 = min2 = float("inf")
    for x in prices:
        if x < min1:
            min1, min2 = x, min1
        elif x < min2:
            min2 = x
    cost = min1 + min2
    return money - cost if cost <= money else money

assert buy_choco_one_pass2([1, 2, 2], 3) == 0
assert buy_choco_one_pass2([3, 2, 3], 3) == 3

# Bài 5
def max_bars(prices, money):
    cnt = 0
    for p in sorted(prices):
        if money < p:
            break
        money -= p
        cnt += 1
    return cnt

assert max_bars([1, 3, 2, 4, 1], 7) == 4
assert max_bars([10, 6, 8, 7, 7, 8], 5) == 0

def count_pairs_affordable(prices, money):
    a = sorted(prices)
    l, r, total = 0, len(a) - 1, 0
    while l < r:
        if a[l] + a[r] <= money:
            total += r - l
            l += 1
        else:
            r -= 1
    return total

assert count_pairs_affordable([1, 2, 2], 3) == 2          # (1,2) hai lần
assert count_pairs_affordable([3, 2, 3], 3) == 0
import itertools
for prices, money in [([5, 1, 4, 2, 2], 6), ([1, 1, 1], 2), ([9, 9], 5)]:
    want = sum(1 for x, y in itertools.combinations(prices, 2) if x + y <= money)
    assert count_pairs_affordable(prices, money) == want
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (hai số nhỏ nhất rồi so với `money`)

```python
from typing import List

class Solution:
    def buyChoco(self, prices: List[int], money: int) -> int:
        min1 = min2 = float("inf")
        for x in prices:
            if x < min1:
                min1, min2 = x, min1
            elif x < min2:
                min2 = x
        cost = min1 + min2
        return money - cost if cost <= money else money
```

**Checklist**

- [ ] Muốn dư nhiều nhất ⇒ trả ít nhất ⇒ chọn **hai thanh rẻ nhất**?
- [ ] Hai thanh phải khác nhau (không `2 * min`)?
- [ ] Kiểm tra `cost <= money` trước khi trừ; không mua được thì trả đúng `money`?
- [ ] Cập nhật `min1`, `min2` bằng `if / elif` (không phải hai `if`)?
- [ ] Đã thử `n = 2`, giá trùng nhau, `cost == money` (dư 0) và `cost == money + 1`?
- [ ] Không làm hỏng mảng đầu vào (nếu dùng `sort()` tại chỗ thì ý thức hậu quả)?
- [ ] Signature đúng `buyChoco(self, prices: List[int], money: int) -> int` và có `from typing import List`?